# RAG LLM LawBot

## Install relevant packages

In [1]:
%%capture

!pip install unsloth
!pip install bitsandbytes
!pip install unsloth_zoo
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install -U sentence-transformers
!pip install -q langchain==0.1.20
!pip install -q langchain-community==0.0.38
!pip install -q chromadb==0.4.24
!pip install -q gradio==4.36.1
!pip install -q pymupdf==1.23.8
!pip install -q sentence-transformers==2.2.2
!pip install youtube_dl
!pip install whisper

## Import all relevant packages throughout this walkthrough

In [2]:
# Modules for fine-tuning
from unsloth import FastLanguageModel
import torch # Import PyTorch
from trl import SFTTrainer # Trainer for supervised fine-tuning (SFT)
from unsloth import is_bfloat16_supported # Checks if the hardware supports bfloat16 precision
# Hugging Face modules
from huggingface_hub import login # Lets you login to API
from transformers import TrainingArguments # Defines training hyperparameters
from datasets import load_dataset # Lets you load fine-tuning datasets
# Import weights and biases
import wandb
# Import kaggle secrets
from kaggle_secrets import UserSecretsClient

import pandas as pd
import re
import string
from sklearn.metrics import precision_score, recall_score
import random
from typing import List, Dict, Tuple

# Import necessary packages
import gradio as gr
import torch
import re
import os
from pathlib import Path
import warnings

# Document processing and retrieval
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma

# Embedding generation using HuggingFace embeddings instead of Ollama
from langchain_community.embeddings import HuggingFaceEmbeddings

# Import kaggle secrets for token management
from kaggle_secrets import UserSecretsClient

from transformers import pipeline

## Import all relevant packages
import torch
import re
import os
import requests
from pathlib import Path
from bs4 import BeautifulSoup
import youtube_dl
import whisper

# Hugging Face modules
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Unsloth and model imports
from unsloth import FastLanguageModel
from transformers import pipeline

# Document processing and retrieval
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.schema import Document

# Alternative embedding using transformers directly
from transformers import AutoTokenizer, AutoModel
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-06-04 23:17:33.155947: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749079053.363312      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749079053.424449      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# warnings.filterwarnings("ignore", message="Unexpected keyword arguments")

## Create API keys and login to Hugging Face and Weights and Biases

In [4]:
user_secrets = UserSecretsClient()
hugging_face_token = user_secrets.get_secret("HF_TOKEN_DEEPSEEK")
wnb_token = user_secrets.get_secret("wnb_token")

login(hugging_face_token)

wandb.login(key=wnb_token)
run = wandb.init(
    project='Fine-tune-DeepSeek-R1-Distill-Llama-8B on LawBot', 
    job_type="training", 
    anonymous="allow"
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: naufalkr394 (naufalkr394-institut-teknologi-sepuluh-nopember) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Loading Model and the Tokenizer

In [5]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

# Load model adapter menggunakan Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="luckysantoso/adapter_deepseek_lawbot",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    token=hugging_face_token,
)

FastLanguageModel.for_inference(model)

/usr/local/lib/python3.11/dist-packages/peft/config.py:162: UserWarning: Unexpected keyword arguments ['alpha_pattern', 'bias', 'corda_config', 'eva_config', 'exclude_modules', 'fan_in_fan_out', 'init_lora_weights', 'layer_replication', 'layers_pattern', 'layers_to_transform', 'loftq_config', 'lora_alpha', 'lora_bias', 'lora_dropout', 'megatron_config', 'megatron_core', 'modules_to_save', 'r', 'rank_pattern', 'target_modules', 'trainable_token_indices', 'use_dora', 'use_rslora'] for class PeftConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


==((====))==  Unsloth 2025.5.10: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/53.0k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/peft/config.py:162: UserWarning: Unexpected keyword arguments ['corda_config', 'trainable_token_indices'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/37.8M [00:00<?, ?B/s]

Unsloth 2025.5.10 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (

## Testing Model on a law use-case before RAG

In [6]:
def test_model_basic():
    """Test basic model functionality"""
    print("\n🧪 Testing basic model functionality...")
    
    prompt_style = """
Di bawah ini adalah instruksi yang menjelaskan tugas, dipasangkan dengan input yang memberikan konteks lebih lanjut.
Tuliskan respons yang menyelesaikan permintaan dengan tepat.
Sebelum menjawab, pikirkan dengan cermat pertanyaan tersebut dan buatlah rangkaian pemikiran langkah demi langkah untuk memastikan respons yang logis dan akurat.

### Instruksi:
Anda adalah seorang ahli hukum dengan pengetahuan tingkat lanjut dalam penalaran hukum, analisis kasus, dan penyusunan dokumen hukum. 
Jawablah pertanyaan hukum berikut ini dengan tepat, berdasarkan peraturan perundang-undangan yang berlaku dan preseden hukum yang relevan.

### Pertanyaan:
{}

### Jawaban:
<think>
"""

    question = """Apa arti dari "berada di bawah Presiden" dalam konteks TNI?"""

    inputs = tokenizer([prompt_style.format(question)], return_tensors="pt").to("cuda")

    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=800,
        use_cache=True,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id else tokenizer.pad_token_id
    )

    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    print("Basic test response:")
    print(response.split("### Jawaban:")[-1] if "### Jawaban:" in response else response)

In [7]:
test_model_basic()


🧪 Testing basic model functionality...
Basic test response:

<think>
Dalam konteks TNI, "berada di bawah Presiden" berarti anggota TNI yang dituduh melanggar hukuman disiplin berada di bawah pengawasan langsung Presiden TNI. Ini berarti Presiden TNI memiliki wewenang penuh untuk mengambil keputusan tentang kasus tersebut.
</think>

Dalam konteks TNI, "berada di bawah Presiden" berarti anggota TNI yang dituduh melanggar hukuman disiplin berada di bawah pengawasan langsung Presiden TNI. Ini berarti Presiden TNI memiliki wewenang penuh untuk mengambil keputusan tentang kasus tersebut.

Penjelasan lebih lanjut:

1. **Wewenang Presiden TNI**: Presiden TNI adalah pimpinan tertinggi dalam TNI dan memiliki otoritas penuh dalam mengurangkan disiplin anggota. Ketika seseorang "berada di bawah Presiden", ini berarti proses pengurangan disiplin tersebut diproses langsung oleh Presiden.

2. **Proses Pengurangan Disiplin**: Ketika anggota dituduh melanggar hukuman disiplin, mereka ditempatkan di ba

## **RAG PEFT Implementation**

In [8]:
def generate_response_peft(question, context="", max_new_tokens=1200):
    """Generate response using PEFT adapter model"""
    
    if context:
        prompt = f"""Di bawah ini adalah instruksi yang menjelaskan tugas, dipasangkan dengan input yang memberikan konteks lebih lanjut.
### Instruksi:
Anda adalah seorang ahli hukum TNI dengan pengetahuan mendalam tentang UU TNI. 
Jawablah pertanyaan berikut berdasarkan pengetahuan Anda yang telah di-fine-tune dengan pasal-pasal UU TNI, 
serta konteks tambahan dari berbagai sumber (buku, artikel, video ahli).
### Konteks dari berbagai sumber:
{context}
### Pertanyaan:
{question}
### Jawaban:
"""
    else:
        prompt = f"""### Instruksi:
Anda adalah seorang ahli hukum TNI dengan pengetahuan mendalam tentang UU TNI.
Jawablah pertanyaan berikut berdasarkan pengetahuan pasal-pasal UU TNI yang telah Anda pelajari.
### Pertanyaan:
{question}
### Jawaban:
"""
    
    try:
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id else tokenizer.pad_token_id
        )
        
        response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        
        if "### Jawaban:" in response:
            answer = response.split("### Jawaban:")[-1].strip()
        else:
            answer = response[len(prompt):].strip()
        
        answer = re.sub(r'```.*?```', '', answer, flags=re.DOTALL).strip()
        answer = re.sub(r'\*\*Final Answer\*\*:.*?(?=\n|$)', '', answer, flags=re.MULTILINE).strip()
        answer = re.sub(r'\*\*Penutup\*\*:.*?(?=\n|$)', '', answer, flags=re.MULTILINE).strip()
        answer = re.sub(r'\*\*Ringkasan\*\*:.*?(?=\n|$)', '', answer, flags=re.MULTILINE).strip()
        
        # Remove repetitive content
        lines = answer.split('\n')
        cleaned_lines = []
        seen_content = set()
        
        for line in lines:
            line = line.strip()
            if line and line not in seen_content:
                cleaned_lines.append(line)
                seen_content.add(line)
        
        answer = '\n'.join(cleaned_lines)
        
        return answer if answer else "Maaf, saya tidak dapat memberikan jawaban yang tepat untuk pertanyaan ini."
        
    except Exception as e:
        print(f"Error generating response: {e}")
        return f"Error generating response: {str(e)}"

In [9]:
class Embeddings:
    def __init__(self, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
        """Use a better multilingual sentence transformer model"""
        print(f"Loading improved embedding model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()
        
        # Add TF-IDF for hybrid scoring
        self.tfidf_vectorizer = TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            stop_words=None,  # Keep Indonesian stopwords handling
            lowercase=True
        )
        self.tfidf_fitted = False
        
    def fit_tfidf(self, texts: List[str]):
        """Fit TF-IDF vectorizer on document corpus"""
        print("Fitting TF-IDF vectorizer...")
        self.tfidf_vectorizer.fit(texts)
        self.tfidf_fitted = True
        
    def embed_documents(self, texts):
        """Embed documents with improved pooling strategy"""
        embeddings = []
        for text in texts:
            embedding = self._get_embedding(text)
            embeddings.append(embedding)
        return embeddings
    
    def embed_query(self, text):
        """Embed query with query-specific preprocessing"""
        # Preprocess query for better matching
        processed_query = self._preprocess_query(text)
        return self._get_embedding(processed_query)
    
    def _preprocess_query(self, query: str) -> str:
        """Preprocess query for better retrieval"""
        query = re.sub(r'\bTNI\b', 'Tentara Nasional Indonesia TNI', query, flags=re.IGNORECASE)
        query = re.sub(r'\bRUU\b', 'Rancangan Undang-Undang RUU', query, flags=re.IGNORECASE)
        query = re.sub(r'\bUU\b', 'Undang-Undang UU', query, flags=re.IGNORECASE)
        query = re.sub(r'\bHAM\b', 'Hak Asasi Manusia HAM', query, flags=re.IGNORECASE)
        
        return query
    
    def _get_embedding(self, text):
        """Get embedding with improved pooling"""
        try:
            text = text[:1024]  # Increased from 512
            inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=1024)
            
            with torch.no_grad():
                outputs = self.model(**inputs)
                
                attention_mask = inputs['attention_mask']
                token_embeddings = outputs.last_hidden_state
                
                input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
                sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
                sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
                embeddings = sum_embeddings / sum_mask
                
                embeddings = embeddings / embeddings.norm(dim=1, keepdim=True)
                
            return embeddings.squeeze().numpy()
        except Exception as e:
            print(f"Error getting embedding: {e}")
            return np.zeros(384)  # Adjust dimension for sentence transformers
    
    def get_hybrid_similarity(self, query: str, texts: List[str], embeddings: np.ndarray) -> np.ndarray:
        """Calculate hybrid similarity using both semantic and lexical matching"""
        if not self.tfidf_fitted:
            return cosine_similarity([self.embed_query(query)], embeddings)[0]
        
        query_embedding = self.embed_query(query)
        semantic_sim = cosine_similarity([query_embedding], embeddings)[0]
        
        try:
            query_tfidf = self.tfidf_vectorizer.transform([query])
            docs_tfidf = self.tfidf_vectorizer.transform(texts)
            lexical_sim = cosine_similarity(query_tfidf, docs_tfidf)[0]
        except:
            lexical_sim = np.zeros(len(texts))
        
        hybrid_sim = 0.7 * semantic_sim + 0.3 * lexical_sim
        
        return hybrid_sim


In [10]:
class QueryExpander:
    def __init__(self):
        self.legal_terms = {
            'tni': ['tentara nasional indonesia', 'militer', 'angkatan bersenjata', 'prajurit', 'angkatan darat/laut/udara'],
            'ruu': ['rancangan undang-undang', 'draft undang-undang', 'usulan undang-undang'],
            'uu': ['undang-undang', 'peraturan', 'legislasi'],
            'ham': ['hak asasi manusia', 'human rights', 'hak dasar'],
            'demokrasi': ['demokratis', 'demokratisasi', 'sistem demokrasi', 'kebebasan sipil'],
            'sipil': ['civilian', 'masyarakat sipil', 'non-militer', 'warga sipil'],
            'masa dinas': ['pensiun', 'usia dinas', 'masa jabatan', 'batas usia'],
            'mk': ['mahkamah konstitusi', 'lembaga peradilan'],
            'komando teritorial': ['koter', 'wilayah militer', 'struktur teritorial'],
            'supremasi sipil': ['kontrol sipil', 'dominasi sipil', 'otoritas sipil'],
            'dpr': ['dewan perwakilan rakyat', 'legislatif', 'parlemen'],
            'kodim': ['komando distrik militer', 'satuan teritorial', 'struktur tni ad'],
            'babinsa': ['bintara pembina desa', 'petugas teritorial', 'perwira lapangan'],
            'pemohon': ['penggugat', 'pelapor', 'pihak yang mengajukan'],
            
        }
    
    def expand_query(self, query: str) -> str:
        """Expand query with related terms"""
        expanded_terms = []
        query_lower = query.lower()
        
        for key, synonyms in self.legal_terms.items():
            if key in query_lower:
                expanded_terms.extend(synonyms[:2])  
        
        if expanded_terms:
            return f"{query} {' '.join(expanded_terms)}"
        return query

In [11]:
class TextSplitter:
    def __init__(self, chunk_size=400, chunk_overlap=100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""]
        )
    
    def split_documents_smart(self, documents: List[Document]) -> List[Document]:
        """Smart document splitting that preserves context"""
        all_chunks = []
        
        for doc in documents:
            source_type = doc.metadata.get('source_type', 'unknown')
            
            if source_type == 'video':
                chunks = self._split_video_transcript(doc)
            elif source_type == 'pdf':
                chunks = self._split_pdf_content(doc)
            else:
                chunks = self.splitter.split_documents([doc])
            
            all_chunks.extend(chunks)
        
        return all_chunks
    
    def _split_video_transcript(self, doc: Document) -> List[Document]:
        """Special handling for video transcripts"""
        video_splitter = RecursiveCharacterTextSplitter(
            chunk_size=300,
            chunk_overlap=50,
            separators=[". ", "! ", "? ", "\n", " "]
        )
        return video_splitter.split_documents([doc])
    
    def _split_pdf_content(self, doc: Document) -> List[Document]:
        """Special handling for PDF content"""
        content = doc.page_content
        
        paragraphs = re.split(r'\n\s*\n', content)
        
        chunks = []
        current_chunk = ""
        
        for paragraph in paragraphs:
            paragraph = paragraph.strip()
            if not paragraph:
                continue
                
            if len(current_chunk) + len(paragraph) > self.chunk_size and current_chunk:
                chunk_doc = Document(
                    page_content=current_chunk.strip(),
                    metadata=doc.metadata.copy()
                )
                chunks.append(chunk_doc)
                
                overlap_words = current_chunk.split()[-20:]  
                current_chunk = " ".join(overlap_words) + " " + paragraph
            else:
                current_chunk += " " + paragraph if current_chunk else paragraph
        
        if current_chunk.strip():
            chunk_doc = Document(
                page_content=current_chunk.strip(),
                metadata=doc.metadata.copy()
            )
            chunks.append(chunk_doc)
        
        return chunks if chunks else self.splitter.split_documents([doc])

In [12]:
class MultiSourceRAG:
    def __init__(self):
        self.documents = []
        self.embeddings = Embeddings()
        self.query_expander = QueryExpander()
        self.text_splitter = TextSplitter(chunk_size=400, chunk_overlap=100)
        
        self.document_embeddings = []
        self.chunk_texts = []
        self.chunk_metadata = []
        self.tfidf_scores = None
        
    def process_pdf(self, pdf_path):
        """Process PDF documents with better text cleaning"""
        print(f"\nProcessing PDF: {pdf_path}")
        try:
            loader = PyMuPDFLoader(pdf_path)
            docs = loader.load()
            
            for doc in docs:
                doc.page_content = self._clean_pdf_text(doc.page_content)
                doc.metadata['source_type'] = 'pdf'
                doc.metadata['source_file'] = os.path.basename(pdf_path)
            
            docs = [doc for doc in docs if len(doc.page_content.strip()) > 50]
            
            self.documents.extend(docs)
            print(f"PDF processed: {len(docs)} pages added")
            return True
        except Exception as e:
            print(f"Error processing PDF: {e}")
            return False
    
    def _clean_pdf_text(self, text: str) -> str:
        """Clean PDF text more thoroughly"""
        # Remove excessive whitespace
        text = re.sub(r'\s+', ' ', text)
        
        # Remove page numbers and headers/footers
        # text = re.sub(r'\b\d+\s*$', '', text, flags=re.MULTILINE)
        # text = re.sub(r'^\s*\d+\s*', '', text, flags=re.MULTILINE)
        
        # Fix common PDF parsing issues
        text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)  # Add space between words
        text = re.sub(r'([.!?])([A-Z])', r'\1 \2', text)  # Add space after punctuation
        
        # Normalize Indonesian text
        text = re.sub(r'\bpasal\s+(\d+)', r'Pasal \1', text, flags=re.IGNORECASE)
        text = re.sub(r'\bundang-undang\b', 'Undang-Undang', text, flags=re.IGNORECASE)
        
        return text.strip()

    def process_web_articles_csv(self, csv_path):
        """Process web articles with better content extraction"""
        print(f"\nProcessing web articles from CSV: {csv_path}")
        try:
            df = pd.read_csv(csv_path)
            print(f"Loaded {len(df)} web articles from CSV")
            
            processed_count = 0
            for idx, row in df.iterrows():
                try:
                    title = str(row['Title']) if 'Title' in row else f"Article {idx+1}"
                    content = str(row['Content']) if 'Content' in row else ""
                    url = str(row['URL']) if 'URL' in row else ""
                    date = str(row['Date']) if 'Date' in row else ""
                    
                    if content and len(content.strip()) > 100:  # Increased minimum length
                        # Clean web content
                        content = self._clean_web_content(content)
                        
                        # Combine title and content for better context
                        full_content = f"{title}\n\n{content}"
                        
                        doc = Document(
                            page_content=full_content,
                            metadata={
                                'source_type': 'web',
                                'title': title,
                                'url': url,
                                'date': date
                            }
                        )
                        self.documents.append(doc)
                        processed_count += 1
                        
                except Exception as e:
                    print(f"Error processing article {idx}: {e}")
                    continue
                    
            print(f"Web articles processed: {processed_count} articles added")
            return True
            
        except Exception as e:
            print(f"Error processing web articles CSV: {e}")
            return False
    
    def _clean_web_content(self, content: str) -> str:
        """Clean web content more thoroughly"""
        # Remove HTML tags if any
        content = re.sub(r'<[^>]+>', '', content)
        
        # Remove excessive whitespace
        content = re.sub(r'\s+', ' ', content)
        
        # Remove common web artifacts
        content = re.sub(r'(Baca juga:|Lihat juga:|Sumber:|Foto:|Video:).*?(?=\n|\.|$)', '', content, flags=re.IGNORECASE)
        
        return content.strip()

    def process_video_transcripts_csv(self, csv_path):
        """Process video transcripts with improved cleaning"""
        print(f"\nProcessing video transcripts from CSV: {csv_path}")
        try:
            df = pd.read_csv(csv_path)
            print(f"Loaded {len(df)} video transcripts from CSV")
            
            processed_count = 0
            for idx, row in df.iterrows():
                try:
                    title = str(row['Title']) if 'Title' in row else f"Video {idx+1}"
                    script = str(row['Script']) if 'Script' in row else ""
                    url = str(row['URL']) if 'URL' in row else ""
                    date = str(row['Date']) if 'Date' in row else ""
                    
                    if script and len(script.strip()) > 100:  # Increased minimum length
                        processed_script = self._preprocess_video_script(script)
                        
                        if len(processed_script.strip()) > 50:
                            # Add title for context
                            full_content = f"Video: {title}\n\n{processed_script}"
                            
                            doc = Document(
                                page_content=full_content,
                                metadata={
                                    'source_type': 'video',
                                    'title': title,
                                    'url': url,
                                    'date': date
                                }
                            )
                            self.documents.append(doc)
                            processed_count += 1
                        
                except Exception as e:
                    print(f"Error processing video {idx}: {e}")
                    continue
                    
            print(f"Video transcripts processed: {processed_count} videos added")
            return True
            
        except Exception as e:
            print(f"Error processing video transcripts CSV: {e}")
            return False
    
    # def _preprocess_video_script(self, script_text):
    #     if not script_text or pd.isna(script_text):
    #         return ""
        
    #     text = str(script_text)
        
    #     # Remove timestamps and noise
    #     text = re.sub(r'\d{1,2}:\d{2}:\d{2}', '', text)
    #     text = re.sub(r'\[.*?\]', '', text)
    #     text = re.sub(r'foreign\s*', '', text, flags=re.IGNORECASE)
    #     text = re.sub(r'(uh|um|eh|mm)\s+', '', text, flags=re.IGNORECASE)
        
    #     # Fix sentence boundaries
    #     text = re.sub(r'([a-z])\s+([A-Z][a-z])', r'\1. \2', text)
    #     text = re.sub(r'(\d{4})\s+([A-Z])', r'\1. \2', text)
        
    #     # Normalize legal terms
    #     text = re.sub(r'pasal\s+(\d+)', r'Pasal \1', text, flags=re.IGNORECASE)
    #     text = re.sub(r'uu\s+tni', 'UU TNI', text, flags=re.IGNORECASE)
    #     text = re.sub(r'ruu\s+tni', 'RUU TNI', text, flags=re.IGNORECASE)
        
    #     # Clean up whitespace
    #     text = re.sub(r'\s+', ' ', text).strip()
        
    #     # Split into sentences and filter quality
    #     sentences = re.split(r'[.!?]+', text)
    #     quality_sentences = []
        
    #     for sentence in sentences:
    #         sentence = sentence.strip()
    #         # Keep sentences that are meaningful length and contain relevant terms
    #         if (len(sentence) > 15 and 
    #             any(term in sentence.lower() for term in ['tni', 'undang', 'pasal', 'militer', 'sipil', 'negara', 'hukum'])):
    #             quality_sentences.append(sentence)
        
    #     return '. '.join(quality_sentences) + '.' if quality_sentences else ""
    
    def create_vector_store(self):
        print(f"\nCreating enhanced vector store from {len(self.documents)} documents...")
        
        if not self.documents:
            print("No documents to process!")
            return False
        
        all_chunks = self.text_splitter.split_documents_smart(self.documents)
        print(f"Created {len(all_chunks)} text chunks with smart splitting")
        
        self.chunk_texts = [chunk.page_content for chunk in all_chunks]
        self.chunk_metadata = [chunk.metadata for chunk in all_chunks]
        
        # Fit TF-IDF on the corpus
        self.embeddings.fit_tfidf(self.chunk_texts)
        
        print("Creating embeddings with improved model...")
        batch_size = 8  
        self.document_embeddings = []
        
        for i in range(0, len(self.chunk_texts), batch_size):
            batch = self.chunk_texts[i:i+batch_size]
            batch_embeddings = self.embeddings.embed_documents(batch)
            self.document_embeddings.extend(batch_embeddings)
            # print(f"  Processed {min(i+batch_size, len(self.chunk_texts))}/{len(self.chunk_texts)} chunks")
        
        print("Vector store created successfully!")
        return True
    
    def retrieve_context(self, question: str, k: int = 5) -> Tuple[str, List[Dict]]:
        """Retrieval with query expansion and hybrid search"""
        if not self.document_embeddings:
            return "", []
        
        try:
            print(f"Search for: {question}")
            
            expanded_query = self.query_expander.expand_query(question)
            print(f"Expanded query: {expanded_query}")
            
            doc_embeddings = np.array(self.document_embeddings)
            similarities = self.embeddings.get_hybrid_similarity(
                expanded_query, self.chunk_texts, doc_embeddings
            )
            
            # Apply dynamic threshold based on query
            threshold = self._get_dynamic_threshold(similarities)
            # print(f"Using dynamic threshold: {threshold:.3f}")
            
            valid_indices = np.where(similarities >= threshold)[0]
            if len(valid_indices) < k:
                top_indices = np.argsort(similarities)[::-1][:k]
            else:
                valid_similarities = similarities[valid_indices]
                top_valid_indices = np.argsort(valid_similarities)[::-1][:k]
                top_indices = valid_indices[top_valid_indices]
            
            contexts = []
            retrieved_docs = []
            
            for idx in top_indices:
                metadata = self.chunk_metadata[idx]
                source_type = metadata.get('source_type', 'unknown')
                source_info = self._format_source_info(metadata, source_type)
                
                context_text = f"{source_info}\n{self.chunk_texts[idx]}"
                contexts.append(context_text)
                
                retrieved_docs.append({
                    'text': self.chunk_texts[idx],
                    'metadata': metadata,
                    'similarity': similarities[idx]
                })
            
            # print(f"Retrieved {len(contexts)} contexts (similarities: {[f'{d[\"similarity\"]:.3f}' for d in retrieved_docs]})")
            return "\n\n".join(contexts), retrieved_docs
            
        except Exception as e:
            print(f"Error in enhanced retrieval: {e}")
            return "", []
    
    def _get_dynamic_threshold(self, similarities: np.ndarray) -> float:
        """Calculate dynamic threshold based on similarity distribution"""
        if len(similarities) == 0:
            return 0.1
        
        mean_sim = np.mean(similarities)
        std_sim = np.std(similarities)
        max_sim = np.max(similarities)
        
        if max_sim > 0.8:
            return max(0.3, mean_sim)
        elif max_sim > 0.5:
            return max(0.2, mean_sim - 0.5 * std_sim)
        else:
            return max(0.1, mean_sim - std_sim)
    
    def _format_source_info(self, metadata: Dict, source_type: str) -> str:
        """Format source information for context"""
        if source_type == 'pdf':
            return f"[PDF: {metadata.get('source_file', 'Unknown')}]"
        elif source_type == 'web':
            title = metadata.get('title', 'Unknown')
            return f"[Artikel: {title[:40]}...]"
        elif source_type == 'video':
            title = metadata.get('title', 'Unknown')
            return f"[Video: {title[:40]}...]"
        else:
            return f"[{source_type.upper()}]"

In [15]:
def calculate_metrics(retrieved_docs, relevant_keywords, expected_topics):
    """Calculate evaluation metrics"""
    if not retrieved_docs:
        return {
            'precision': 0.0,
            'recall': 0.0,
            'f1': 0.0,
            'relevance_score': 0.0,
            'diversity_score': 0.0
        }
    
    relevance_scores = []
    source_types = []
    
    for doc in retrieved_docs:
        text_lower = doc['text'].lower()
        
        keyword_score = 0
        for keyword in relevant_keywords:
            if keyword.lower() in text_lower:
                keyword_score += 1
            else:
                words = keyword.lower().split()
                if len(words) > 1 and all(word in text_lower for word in words):
                    keyword_score += 0.5
        
        keyword_score = min(keyword_score / len(relevant_keywords), 1.0)
        
        topic_score = 0
        for topic in expected_topics:
            topic_words = topic.lower().split()
            matches = sum(1 for word in topic_words if word in text_lower)
            if matches >= len(topic_words) * 0.7:  
                topic_score += 1
        
        topic_score = min(topic_score / len(expected_topics), 1.0)
        
        relevance = 0.4 * keyword_score + 0.4 * topic_score + 0.2 * min(doc.get('similarity', 0.0), 1.0)
        relevance_scores.append(relevance)
        
        source_types.append(doc['metadata'].get('source_type', 'unknown'))
    
    relevant_count = sum(1 for score in relevance_scores if score > 0.3)
    precision = relevant_count / len(retrieved_docs)
    
    estimated_total_relevant = max(3, len(relevant_keywords))  
    recall = min(relevant_count / estimated_total_relevant, 1.0)
    
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    unique_sources = len(set(source_types))
    max_possible_sources = 3  # pdf, web, video
    diversity_score = unique_sources / max_possible_sources
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'relevance_score': np.mean(relevance_scores),
        'diversity_score': diversity_score
    }

## Extract & Preprocess RAG Dataset

In [16]:
rag_system = MultiSourceRAG()

Loading improved embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

In [17]:
PDF_SOURCES = [
    "/kaggle/input/rag-pdf/Permohonan_4288_8190_Permohonan_redact.pdf",
    "/kaggle/input/rag-pdf/Petisi-Revisi-UU-TNI-.pdf", 
    "/kaggle/input/rag-pdf/naskah-akademik.pdf",
    "/kaggle/input/rag-pdf/wiraedsus2019-web.pdf"
]
WEB_CSV_PATH = "/kaggle/input/rag-web-scrap/detik_articles.csv"
VIDEO_CSV_PATH = ""  

In [18]:
pdf_processed = 0
for pdf_path in PDF_SOURCES:
    if os.path.exists(pdf_path):
        if rag_system.process_pdf(pdf_path):
            pdf_processed += 1
        else:
            print(f"Failed to process: {pdf_path}")
    else:
        print(f"PDF not found: {pdf_path}")

print(f"Successfully processed {pdf_processed}/{len(PDF_SOURCES)} PDF files")

if os.path.exists(WEB_CSV_PATH):
    if rag_system.process_web_articles_csv(WEB_CSV_PATH):
        print("Web articles processed successfully")
    else:
        print("Failed to process web articles")
else:
    print(f"Web articles CSV not found: {WEB_CSV_PATH}")

# if VIDEO_CSV_PATH and os.path.exists(VIDEO_CSV_PATH):
#     if rag_system.process_video_transcripts_csv(VIDEO_CSV_PATH):
#         print("Video transcripts processed successfully")
#     else:
#         print("Failed to process video transcripts")
# else:
#     print("No video transcript source provided")



Processing PDF: /kaggle/input/rag-pdf/Permohonan_4288_8190_Permohonan_redact.pdf
PDF processed: 24 pages added

Processing PDF: /kaggle/input/rag-pdf/Petisi-Revisi-UU-TNI-.pdf
PDF processed: 11 pages added

Processing PDF: /kaggle/input/rag-pdf/naskah-akademik.pdf
PDF processed: 28 pages added

Processing PDF: /kaggle/input/rag-pdf/wiraedsus2019-web.pdf
PDF processed: 56 pages added
Successfully processed 4/4 PDF files

Processing web articles from CSV: /kaggle/input/rag-web-scrap/detik_articles.csv
Loaded 334 web articles from CSV
Web articles processed: 334 articles added
Web articles processed successfully


In [19]:
if rag_system.documents:
    print(f"\n🔧 Creating vector store from {len(rag_system.documents)} documents...")
    
    if rag_system.create_vector_store():
        
        source_stats = {}
        for doc in rag_system.documents:
            source_type = doc.metadata.get('source_type', 'unknown')
            source_stats[source_type] = source_stats.get(source_type, 0) + 1
        
        chunk_stats = {}
        for metadata in rag_system.chunk_metadata:
            source_type = metadata.get('source_type', 'unknown')
            chunk_stats[source_type] = chunk_stats.get(source_type, 0) + 1
        
        print(f"   Total Documents: {len(rag_system.documents)}")
        print(f"   Total Chunks: {len(rag_system.chunk_texts)}")
        print(f"   Embedding Dimension: {len(rag_system.document_embeddings[0]) if rag_system.document_embeddings else 0}")
        
        print(f"\nBy Source Type:")
        for source_type in source_stats:
            docs = source_stats.get(source_type, 0)
            chunks = chunk_stats.get(source_type, 0)
            print(f"   {source_type.upper()}: {docs} documents → {chunks} chunks")
    else:
        print("Failed to create vector store")
else:
    print("No documents processed. Please check your data sources.")
    print("Make sure the file paths are correct and files exist.")



🔧 Creating vector store from 453 documents...

Creating enhanced vector store from 453 documents...
Created 3319 text chunks with smart splitting
Fitting TF-IDF vectorizer...
Creating embeddings with improved model...
Vector store created successfully!
   Total Documents: 453
   Total Chunks: 3319
   Embedding Dimension: 384

By Source Type:
   PDF: 119 documents → 119 chunks
   WEB: 334 documents → 3200 chunks


## RAG Evaluation

In [24]:
def test_system():
    if len(rag_system.document_embeddings) == 0:
        print("Vector store not available. Cannot run tests.")
        return
    
    test_queries = [
        "Mengapa RUU TNI 2025 dianggap kontroversial oleh masyarakat sipil?",
        "Apa dampak perpanjangan masa dinas TNI terhadap demokrasi?", 
        "Bagaimana tanggapan ahli hukum terhadap RUU TNI?",
        "Jelaskan hubungan TNI dan masyarakat sipil dalam RUU TNI",
        "Apa saja pasal kontroversial dalam RUU TNI?"
    ]
    
    for i, query in enumerate(test_queries, 1):
        print(f"\nTest Query {i}: {query}")
        print("-" * 50)
        
        # Test retrieval only
        context, docs = rag_system.retrieve_context(query, k=3)
        
        if docs:
            print(f"Retrieved {len(docs)} documents")
            avg_similarity = np.mean([doc['similarity'] for doc in docs])
            print(f"Average similarity: {avg_similarity:.3f}")
            
            source_types = [doc['metadata'].get('source_type') for doc in docs]
            unique_sources = len(set(source_types))
            print(f"Source diversity: {unique_sources}/3 types")
        else:
            print("No documents retrieved")


In [25]:
def evaluate_system():
    if len(rag_system.document_embeddings) == 0:
        print("Vector store not available. Cannot run evaluation.")
        return
    
    evaluation_queries = [
        {
            "query": "RUU TNI kontroversial",
            "keywords": ["RUU", "TNI", "kontroversial", "masyarakat sipil", "demokrasi"],
            "topics": ["perpanjangan masa dinas", "peran TNI", "kontrol sipil"]
        },
        {
            "query": "perpanjangan masa dinas TNI",
            "keywords": ["masa dinas", "perpanjangan", "TNI", "pensiun", "usia"],
            "topics": ["reformasi TNI", "profesionalisme", "regenerasi"]
        },
        {
            "query": "hubungan TNI masyarakat sipil",
            "keywords": ["TNI", "sipil", "hubungan", "kontrol", "demokrasi"],
            "topics": ["civil-military relations", "demokratisasi", "reformasi"]
        }
    ]
    
    overall_scores = []
    
    for i, eval_item in enumerate(evaluation_queries, 1):
        query = eval_item["query"]
        keywords = eval_item["keywords"]
        topics = eval_item["topics"]
        
        print(f"\nEvaluation {i}: {query}")
        print("-" * 40)
        
        # Test different k values
        for k in [3, 5, 7]:
            context, docs = rag_system.retrieve_context(query, k=k)
            
            if docs:
                metrics = calculate_metrics(docs, keywords, topics)
                overall_scores.append(metrics)
                
                print(f"k={k}: P={metrics['precision']:.3f}, R={metrics['recall']:.3f}, "
                      f"f1={metrics['f1']:.3f}, Rel={metrics['relevance_score']:.3f}, "
                      f"Div={metrics['diversity_score']:.3f}")
            else:
                print(f"k={k}: No results")
    
    # Calculate overall performance
    if overall_scores:
        avg_metrics = {}
        for key in overall_scores[0].keys():
            avg_metrics[key] = np.mean([score[key] for score in overall_scores])
        
        print(f"\nOVERALL RAG PERFORMANCE:")
        print("-" * 40)
        for metric, score in avg_metrics.items():
            print(f"{metric.replace('_', ' ').title()}: {score:.3f}")
        
        avg_f1 = avg_metrics['f1']

In [26]:
if rag_system.documents:
    print(f"   Documents: {len(rag_system.documents)}")
    print(f"   Chunks: {len(rag_system.chunk_texts) if hasattr(rag_system, 'chunk_texts') else 'Not created'}")
    print(f"   Embeddings: {len(rag_system.document_embeddings) if hasattr(rag_system, 'document_embeddings') else 'Not created'}")
    
    if len(rag_system.document_embeddings) > 0:
        test_system()
        
        evaluate_system()
        
    else:
        print("\nVector store not ready. Run rag_system.create_vector_store() first.")
else:
    print("\nNo documents loaded. Please check your data sources.")

   Documents: 453
   Chunks: 3319
   Embeddings: 3319

Test Query 1: Mengapa RUU TNI 2025 dianggap kontroversial oleh masyarakat sipil?
--------------------------------------------------
Search for: Mengapa RUU TNI 2025 dianggap kontroversial oleh masyarakat sipil?
Expanded query: Mengapa RUU TNI 2025 dianggap kontroversial oleh masyarakat sipil? tentara nasional indonesia militer rancangan undang-undang draft undang-undang undang-undang peraturan civilian masyarakat sipil
Retrieved 3 documents
Average similarity: 0.683
Source diversity: 1/3 types

Test Query 2: Apa dampak perpanjangan masa dinas TNI terhadap demokrasi?
--------------------------------------------------
Search for: Apa dampak perpanjangan masa dinas TNI terhadap demokrasi?
Expanded query: Apa dampak perpanjangan masa dinas TNI terhadap demokrasi? tentara nasional indonesia militer demokratis demokratisasi pensiun usia dinas
Retrieved 3 documents
Average similarity: 0.555
Source diversity: 2/3 types

Test Query 3: Bagai

## RAG Model Output

In [27]:
def format_answer(answer):
    """Format the answer for better readability"""
    
    answer = re.sub(r'### Jawaban akhir:.*?(?=\n\n|\Z)', '', answer, flags=re.DOTALL)
    answer = re.sub(r'\*\*Final Answer\*\*.*?(?=\n\n|\Z)', '', answer, flags=re.DOTALL)
    answer = re.sub(r'```.*?```', '', answer, flags=re.DOTALL)
    
    paragraphs = [p.strip() for p in answer.split('\n\n') if p.strip()]
    
    seen = set()
    unique_paragraphs = []
    for p in paragraphs:
        if p not in seen and len(p) > 10:  
            seen.add(p)
            unique_paragraphs.append(p)
    
    formatted = '\n\n'.join(unique_paragraphs)
    
    # Clean up formatting
    formatted = re.sub(r'\n{3,}', '\n\n', formatted)  # Max 2 consecutive newlines
    formatted = re.sub(r'(\d+)\.\s*\*\*', r'\n\1. **', formatted)  # Fix numbered lists
    
    return formatted.strip()

In [28]:
def generate_model_response(question, use_rag=True, k=5, show_sources=True, verbose=False):
    """Generate response using Enhanced RAG system + PEFT model with better formatting"""
    
    print(f"\n{'='*80}")
    print(f"🔍 PERTANYAAN: {question}")
    print(f"{'='*80}")
    
    context = ""
    retrieved_docs = []
    
    if use_rag and len(rag_system.document_embeddings) > 0:
        if verbose:
            print(f"📚 Mencari konteks relevan (k={k})...")
        
        context, retrieved_docs = rag_system.retrieve_context(question, k=k)
        
        if retrieved_docs:
            if verbose:
                print(f"Berhasil menemukan {len(retrieved_docs)} konteks relevan")
            
            if show_sources:
                print(f"\n📋 SUMBER REFERENSI:")
                print("-" * 50)
                for i, doc in enumerate(retrieved_docs):
                    source_type = doc['metadata'].get('source_type', 'unknown')
                    similarity = doc['similarity']
                    
                    if source_type == 'pdf':
                        source_name = doc['metadata'].get('source_file', 'Unknown PDF')
                        source_info = f"{source_name}"
                    elif source_type == 'web':
                        source_name = doc['metadata'].get('title', 'Unknown Article')
                        if len(source_name) > 60:
                            source_name = source_name[:60] + "..."
                        source_info = f"{source_name}"
                    elif source_type == 'video':
                        source_name = doc['metadata'].get('title', 'Unknown Video')
                        if len(source_name) > 60:
                            source_name = source_name[:60] + "..."
                        source_info = f"{source_name}"
                    else:
                        source_info = f"Unknown Source"
                    
                    print(f"   {i+1}. {source_info} (Relevansi: {similarity:.1%})")
        else:
            if verbose:
                print("Tidak menemukan konteks yang relevan")
    else:
        if verbose:
            print("RAG tidak aktif atau vector store tidak tersedia")
    
    print(f"\nJAWABAN:")
    print("-" * 80)
    
    if verbose:
        print("Menggunakan model fine-tuned untuk menghasilkan jawaban...")
    
    answer = generate_response_peft(question, context)
    
    # Format the answer nicely
    formatted_answer = format_answer(answer)
    print(formatted_answer)
    
    print(f"\n{' '*80}")
    
    return formatted_answer, retrieved_docs

In [34]:
print("\n" + "="*60)
print("TESTING CHATBOT UU TNI WITH RAG")
print("="*60)

# Test questions
test_questions = [
    "Mengapa RUU TNI 2025 dianggap kontroversial oleh masyarakat sipil?",
    "Apa dampak perpanjangan masa dinas TNI terhadap demokrasi?", 
    "Bagaimana tanggapan ahli hukum terhadap RUU TNI?",
    "Jelaskan hubungan TNI dan masyarakat sipil dalam RUU TNI",
    "Apa saja pasal yang diubah pada revisi UU TNI 2025?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*20} TEST {i} {'='*20}")
    print(f"Pertanyaan: {question}")
    print("\nJawaban (dengan Multi-Source RAG):")
    print("-" * 50)
    
    answer = generate_model_response(question, use_rag=True)
    # print(answer)
    
    print("\n" + " "*60)


TESTING CHATBOT UU TNI WITH RAG

==================== TEST 1 ====================
Pertanyaan: Mengapa RUU TNI 2025 dianggap kontroversial oleh masyarakat sipil?

Jawaban (dengan Multi-Source RAG):
--------------------------------------------------

🔍 PERTANYAAN: Mengapa RUU TNI 2025 dianggap kontroversial oleh masyarakat sipil?
Search for: Mengapa RUU TNI 2025 dianggap kontroversial oleh masyarakat sipil?
Expanded query: Mengapa RUU TNI 2025 dianggap kontroversial oleh masyarakat sipil? tentara nasional indonesia militer rancangan undang-undang draft undang-undang undang-undang peraturan civilian masyarakat sipil

📋 SUMBER REFERENSI:
--------------------------------------------------
   1. Pendemo Protes Pembahasan RUU TNI: Kita Nggak Punya Draf Res... (Relevansi: 68.9%)
   2. Pakar di Jogja Kritik Pengesahan UU TNI, Singgung Kemenangan... (Relevansi: 68.1%)
   3. Pakar Hukum UB Singgung Orde Baru Usai UU TNI Disahkan (Relevansi: 67.9%)
   4. Puan: Perubahan UU TNI Tetap Berlandaskan 

In [131]:
# def create_gradio_interface():
#     """Create a Gradio web interface for the chatbot"""
    
#     def chat_interface(message, history):
#         try:
#             response = generate_response(message, use_rag=True)
#             history.append((message, response))
#             return history, ""
#         except Exception as e:
#             error_msg = f"Error: {str(e)}"
#             history.append((message, error_msg))
#             return history, ""
    
#     with gr.Blocks(title="RAG LawBot TNI") as interface:
#         gr.Markdown("# RAG LawBot TNI")
#         gr.Markdown("Chatbot hukum TNI dengan fine-tuned model dan RAG")
        
#         chatbot = gr.Chatbot(height=400)
#         msg = gr.Textbox(placeholder="Tanyakan sesuatu tentang UU TNI...")
#         clear = gr.Button("Clear")
        
#         msg.submit(chat_interface, [msg, chatbot], [chatbot, msg])
#         clear.click(lambda: [], None, chatbot, queue=False)
    
#     return interface

In [132]:
# interface = create_simple_gradio_interface()
# interface.launch(share=True)